# CS4241 - Introduction to Artificial Intelligence
## Part A: Data Engineering & Preparation

**Name:** Maureen Amago  
**Index Number:** 10022200180

---
## Step 1: Install Required Libraries
Run this cell first to make sure all needed libraries are installed.

In [ ]:
# Name: Maureen Amago | Index: 10022200180
import subprocess
subprocess.run(['pip', 'install', 'pypdf', '--quiet'], check=True)
print('All libraries are ready!')

---
## Step 2: Import Libraries

In [ ]:
# Name: Maureen Amago | Index: 10022200180
import pandas as pd
import re
import numpy as np
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries imported successfully!')

---
## Step 3: Data Cleaning

We have two datasets:
1. `Ghana_Election_Result.csv` - Ghana election results (1992-2020)
2. `2025-Budget-Statement-and-Economic-Policy_v4.pdf` - Ghana 2025 Budget document

We will clean both and use the **PDF** for the chunking and retrieval tasks.

### 3a. Clean the CSV File

In [ ]:
# Name: Maureen Amago | Index: 10022200180

# Load the CSV file
df = pd.read_csv('Ghana_Election_Result.csv')

print('=== BEFORE CLEANING ===')
print(f'Shape: {df.shape}')
print('\nFirst 5 rows:')
display(df.head())
print('\nMissing values per column:')
print(df.isnull().sum())

In [ ]:
# Name: Maureen Amago | Index: 10022200180

# --- Clean the CSV ---

# 1. Strip whitespace from column names and make them lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# 2. Drop rows where ALL columns are empty (blank rows)
df_cleaned = df.dropna(how='all')

# 3. Clean the 'votes(%)' column: remove the % sign and convert to float
df_cleaned = df_cleaned.copy()
df_cleaned['votes(%)'] = df_cleaned['votes(%)'].astype(str).str.replace('%', '').str.strip().astype(float)

# 4. Rename the column to something cleaner
df_cleaned.rename(columns={'votes(%)': 'vote_percentage'}, inplace=True)

print('=== AFTER CLEANING ===')
print(f'Shape: {df_cleaned.shape}')
print('\nCleaned columns:', list(df_cleaned.columns))
print('\nFirst 5 rows:')
display(df_cleaned.head())
print('\nMissing values after cleaning:')
print(df_cleaned.isnull().sum())

### 3b. Extract and Clean the PDF File

In [ ]:
# Name: Maureen Amago | Index: 10022200180

# Extract text from the PDF (first 30 pages to keep it manageable)
reader = PdfReader('2025-Budget-Statement-and-Economic-Policy_v4.pdf')
total_pages = len(reader.pages)
print(f'Total pages in PDF: {total_pages}')

raw_text = ''
pages_to_read = min(30, total_pages)  # read up to 30 pages

for i in range(pages_to_read):
    page_text = reader.pages[i].extract_text()
    if page_text:
        raw_text += page_text + ' '

print(f'\nRaw text extracted: {len(raw_text)} characters')
print('\nRaw text sample (first 300 chars):')
print(raw_text[:300])

In [ ]:
# Name: Maureen Amago | Index: 10022200180

# --- Clean the extracted PDF text ---

# 1. Replace multiple whitespace characters (spaces, newlines, tabs) with a single space
clean_text = re.sub(r'\s+', ' ', raw_text)

# 2. Remove special/garbage characters that come from PDF extraction
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', clean_text)  # remove non-ASCII characters

# 3. Strip leading/trailing spaces
clean_text = clean_text.strip()

print(f'Cleaned text length: {len(clean_text)} characters')
print('\nCleaned text sample (first 500 chars):')
print(clean_text[:500])

---
## Step 4: Chunking Strategy Design

### Why do we chunk?
When building an AI system (like a Question & Answer chatbot), we cannot feed an entire 300-page document into a model at once. We must split it into smaller, manageable pieces called **chunks**. The AI then searches through these chunks to find the most relevant one for a given question.

### Our Two Strategies

| Strategy | Chunk Size | Overlap | Rationale |
|---|---|---|---|
| **Strategy 1** (Recommended) | 500 characters | 50 characters | Small & focused. Each chunk covers roughly 2-4 sentences. Ideal for finding specific facts. The 50-char overlap ensures no sentence is cut off at the boundary. |
| **Strategy 2** (For Comparison) | 2000 characters | 0 characters | Large & broad. Each chunk covers many paragraphs. Less focused but captures more context. No overlap means information at boundaries may be lost. |

### Justification for Strategy 1 (chunk_size=500, overlap=50)
- **500 characters** ≈ 3-4 sentences, which is enough to capture a complete idea without being too long.
- **50-character overlap** ensures that if a sentence is split between two chunks, both chunks still contain enough context for the AI to understand it correctly.

---
## Step 5: Implement Chunking Strategies

In [ ]:
# Name: Maureen Amago | Index: 10022200180

def chunk_text(text, chunk_size, overlap):
    """
    Splits a text into overlapping chunks.

    Parameters:
    - text: the full cleaned text string
    - chunk_size: number of characters per chunk
    - overlap: number of characters to share between consecutive chunks

    Returns:
    - A list of text chunks
    """
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        # Move forward by (chunk_size - overlap) so the next chunk starts
        # slightly before where this one ended
        start += chunk_size - overlap

    return chunks


# --- Apply Strategy 1: Small chunks WITH overlap ---
chunks_strategy_1 = chunk_text(clean_text, chunk_size=500, overlap=50)

# --- Apply Strategy 2: Large chunks with NO overlap ---
chunks_strategy_2 = chunk_text(clean_text, chunk_size=2000, overlap=0)

print(f'Strategy 1 (size=500, overlap=50) => {len(chunks_strategy_1)} chunks created')
print(f'Strategy 2 (size=2000, overlap=0) => {len(chunks_strategy_2)} chunks created')

print('\n--- Example Chunk from Strategy 1 (Chunk #1) ---')
print(chunks_strategy_1[0])

print('\n--- Example Chunk from Strategy 2 (Chunk #1, first 500 chars) ---')
print(chunks_strategy_2[0][:500], '...')

---
## Step 6: Comparative Analysis — Chunking Impact on Retrieval Quality

We will now test both strategies by running a **retrieval query**: we will search for `"economic growth and revenue"` and see which strategy returns the most focused, relevant result.

We use **TF-IDF + Cosine Similarity** — a standard AI technique for matching text.

In [ ]:
# Name: Maureen Amago | Index: 10022200180

def retrieve_best_chunk(query, chunks, top_k=2):
    """
    Given a query, finds the most relevant chunks from the list.
    Uses TF-IDF vectorization and Cosine Similarity.
    """
    vectorizer = TfidfVectorizer(stop_words='english')

    # Combine all chunks + the query into one list for vectorization
    all_texts = chunks + [query]
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    # Query is the last vector; everything else is a chunk
    query_vector = tfidf_matrix[-1]
    chunk_vectors = tfidf_matrix[:-1]

    # Compute similarity between the query and each chunk
    similarities = cosine_similarity(query_vector, chunk_vectors).flatten()

    # Get the indices of the top-k most similar chunks
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    results = []
    for i in top_indices:
        results.append({
            'chunk_index': i,
            'similarity_score': round(similarities[i], 4),
            'chunk_text': chunks[i]
        })
    return results


# --- Run the Retrieval Test ---
query = "economic growth and revenue"

print(f'Query: "{query}"')
print('=' * 60)

print('\n>>> STRATEGY 1 RESULT (chunk_size=500, overlap=50)')
print('-' * 60)
results_1 = retrieve_best_chunk(query, chunks_strategy_1, top_k=1)
for r in results_1:
    print(f"Chunk Index : {r['chunk_index']}")
    print(f"Similarity  : {r['similarity_score']}")
    print(f"Chunk Text  :\n{r['chunk_text']}")

print('\n>>> STRATEGY 2 RESULT (chunk_size=2000, overlap=0)')
print('-' * 60)
results_2 = retrieve_best_chunk(query, chunks_strategy_2, top_k=1)
for r in results_2:
    print(f"Chunk Index : {r['chunk_index']}")
    print(f"Similarity  : {r['similarity_score']}")
    print(f"Chunk Text (first 500 chars) :\n{r['chunk_text'][:500]} ...")

---
## Step 7: Conclusion & Analysis Summary

### Comparative Analysis Results

| | Strategy 1 (500 chars, 50 overlap) | Strategy 2 (2000 chars, no overlap) |
|---|---|---|
| **Number of Chunks** | More chunks | Fewer chunks |
| **Chunk Focus** | Highly focused (3-4 sentences) | Very broad (many paragraphs) |
| **Retrieval Quality** | **Better** — returns tight, relevant text | Worse — returns large blocks with noise |
| **Boundary Loss** | Minimal (overlap protects context) | High (no overlap = context can be lost) |

### Final Recommendation
**Strategy 1** (chunk_size=500, overlap=50) is the better design choice for an AI retrieval system because:
1. It returns **focused** and **relevant** text for a given query.
2. The **overlap** prevents any important sentences from being cut and lost between chunks.
3. It is more efficient for AI models which have a limited context window.